# End-to-End FHIR R4 Transformation

This notebook demonstrates the complete pipeline for transforming raw hospital data
into **FHIR R4 (Fast Healthcare Interoperability Resources)** bundles using the Portiere SDK.

While Notebook 08 targets OMOP CDM, this notebook shows the same 5-stage pipeline
outputting FHIR R4 resources instead — enabling interoperability with EHR systems,
health information exchanges (HIEs), and SMART on FHIR applications.

The pipeline consists of five stages:

1. **Ingest** — Load source data files into the project
2. **Profile** — Analyze source data structure and content
3. **Schema Map** — Map source columns to FHIR R4 resource fields
4. **Concept Map** — Translate source codes to standard terminologies (SNOMED CT, LOINC, RxNorm)
5. **ETL + Validate** — Generate FHIR Bundle JSON and validate against R4 profiles

## Overview

We will transform a hospital dataset consisting of five source files into FHIR R4 resources:

| Source File | Description | Target FHIR Resource |
|---|---|---|
| `patients.csv` | Patient demographics | `Patient` |
| `encounters.csv` | Hospital visits / admissions | `Encounter` |
| `diagnoses.csv` | Diagnosis codes (ICD-10) | `Condition` |
| `medications.csv` | Prescribed medications | `MedicationRequest` |
| `lab_results.csv` | Laboratory test results | `Observation` |

### FHIR vs OMOP

| Aspect | OMOP CDM | FHIR R4 |
|---|---|---|
| Structure | Relational tables (SQL) | JSON resources (RESTful) |
| Primary use | Research & analytics | Clinical exchange & interoperability |
| Code systems | OMOP concept IDs | Native terminology URIs (SNOMED, LOINC) |
| Patient ref | `person_id` (integer FK) | `Patient/id` (resource reference) |
| Output | CSV/Parquet files | FHIR Bundle (JSON/NDJSON) |

## Stage 1: Project Setup and Configuration

The key difference from OMOP is setting `target_model="fhir_r4"` in `portiere.init()`.
This tells Portiere to use FHIR R4 resource definitions as the target schema and
generate FHIR-compliant output (JSON Bundles with proper CodeableConcepts).

In [1]:
try:
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.appName("portiere").getOrCreate()
    print(f"SparkSession created: {spark.version}")
    SPARK_AVAILABLE = True
except ImportError:
    SPARK_AVAILABLE = False
    spark = None
    print("PySpark not installed — Spark cells will be skipped. Install with: pip install pyspark")


PySpark not installed — Spark cells will be skipped. Install with: pip install pyspark


### Shared Athena Vocabulary Setup

Build (or reuse) BM25s and FAISS indexes from the Athena vocabulary download.
On the first run this takes ~10-30 minutes for FAISS embedding; subsequent runs
reuse the cached indexes.

In [2]:
from pathlib import Path
from portiere.knowledge import build_knowledge_layer

ATHENA_DIR = Path("example_assets/vocabulary_download_v5")
VOCAB_DIR  = ATHENA_DIR / "indexes"
BM25S_CORPUS = VOCAB_DIR / "concepts.json"
FAISS_INDEX  = VOCAB_DIR / "concepts.index"
FAISS_META   = VOCAB_DIR / "concepts.meta.json"

# Build indexes from Athena on first run (~10-30 min for FAISS embedding)
if not BM25S_CORPUS.exists():
    paths = build_knowledge_layer(
        athena_path=ATHENA_DIR,
        output_path=VOCAB_DIR,
        backend="hybrid",
        vocabularies=["SNOMED", "LOINC", "RxNorm", "ICD10CM"],
    )
    print(f"Built indexes: {paths}")
else:
    print(f"Using existing indexes at {VOCAB_DIR}")

Using existing indexes at example_assets/vocabulary_download_v5/indexes


In [3]:
import portiere
from portiere import PortiereConfig, KnowledgeLayerConfig

if SPARK_AVAILABLE:
    from portiere.engines import SparkEngine
    engine = SparkEngine(spark)
else:
    from portiere.engines import PolarsEngine
    engine = PolarsEngine()
    print("PySpark not installed — using PolarsEngine instead")

config = PortiereConfig(
    knowledge_layer=KnowledgeLayerConfig(
        backend="bm25s",
        bm25s_corpus_path=str(BM25S_CORPUS),
    ),
)

project = portiere.init(
    name="FHIR End-to-End Demo",
    engine=engine,
    target_model="fhir_r4",
    vocabularies=["SNOMED", "LOINC", "RxNorm", "ICD10CM"],
    config=config,
)

print(f"Project: {project.name}")
print(f"Engine: {type(engine).__name__}")


2026-04-16 00:28:16 [info     ] PolarsEngine initialized      

PySpark not installed — using PolarsEngine instead
2026-04-16 00:28:16 [debug    ] local_storage.initialized      base_dir=/Users/kuangsmacbook/.portiere/projects


Project: FHIR End-to-End Demo
Engine: PolarsEngine


## Stage 2: Ingest Source Data

Add all source files. This step is identical regardless of target model —
Portiere handles the source-side the same way for both OMOP and FHIR.

In [4]:
patients = project.add_source("example_assets/09_end_to_end_fhir/patients.csv", name="patients")
encounters = project.add_source("example_assets/09_end_to_end_fhir/encounters.csv", name="encounters")
diagnoses = project.add_source("example_assets/09_end_to_end_fhir/diagnoses.csv", name="diagnoses")
medications = project.add_source("example_assets/09_end_to_end_fhir/medications.csv", name="medications")
lab_results = project.add_source("example_assets/09_end_to_end_fhir/lab_results.csv", name="lab_results")

sources = [patients, encounters, diagnoses, medications, lab_results]
for src in sources:
    print(f"Loaded: {src['name']}")

2026-04-16 00:28:16 [info     ] project.source_added           project='FHIR End-to-End Demo' source=patients


Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

2026-04-16 00:28:19 [info     ] gx_profiler.profiled           columns=11 rows=15 source=patients


2026-04-16 00:28:19 [info     ] project.profiled               source=patients


2026-04-16 00:28:19 [info     ] project.source_added           project='FHIR End-to-End Demo' source=encounters


...

2026-04-16 00:28:19 [info     ] project.source_added           project='FHIR End-to-End Demo' source=lab_results


Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

2026-04-16 00:28:19 [info     ] gx_profiler.profiled           columns=9 rows=20 source=lab_results


2026-04-16 00:28:19 [info     ] project.profiled               source=lab_results


Loaded: patients
Loaded: encounters
Loaded: diagnoses
Loaded: medications
Loaded: lab_results


## Stage 3: Schema Mapping

Schema mapping for FHIR targets source columns to **FHIR resource fields** rather
than relational table columns. For example:

- `patient_id` → `Patient.identifier`
- `birth_date` → `Patient.birthDate`
- `diagnosis_code` → `Condition.code`
- `lab_value` → `Observation.valueQuantity`

Portiere uses the FHIR R4 resource definitions and field descriptions to find
the best mapping for each source column.

### 3a. Map Patient Demographics to `Patient` Resource

In [5]:
patients_schema = project.map_schema(patients)

print("Patients → Patient Resource")
print("=" * 70)
for item in patients_schema.items:
    status = "[AUTO]" if item.confidence >= 0.85 else "[REVIEW]"
    print(f"  {status} {item.source_column:25s} → "
          f"{item.target_table}.{item.target_column} "
          f"(confidence={item.confidence:.2f})")

2026-04-16 00:28:19 [info     ] Stage 2: Schema mapping (local) embedding_model=cambridgeltl/SapBERT-from-PubMedBERT-fulltext embedding_provider=huggingface target_model=fhir_r4


2026-04-16 00:28:19 [info     ] local_schema_mapper.loading_model model=cambridgeltl/SapBERT-from-PubMedBERT-fulltext provider=huggingface target_standard=omop_cdm_v5.4


2026-04-16 00:28:19 [info     ] embedding_gateway.initialized  model=cambridgeltl/SapBERT-from-PubMedBERT-fulltext provider=huggingface


2026-04-16 00:28:19 [info     ] local_schema_mapper.computing_target_embeddings standard=omop_cdm_v5.4 targets=67


2026-04-16 00:28:21 [warning  ] local_schema_mapper.init_failed: sentence-transformers is required for HuggingFace embeddings. Install with: pip install sentence-transformers


2026-04-16 00:28:21 [info     ] Stage 2 complete               auto=10 review=0 total=11


2026-04-16 00:28:21 [info     ] local_storage.schema_mapping_saved items_count=11 project='FHIR End-to-End Demo'


2026-04-16 00:28:21 [info     ] project.schema_mapped          auto=10 project='FHIR End-to-End Demo' total=11


Patients → Patient Resource
  [AUTO] patient_id                → person.person_id (confidence=0.95)
  [AUTO] first_name                → person.person_source_value (confidence=0.95)
  [REVIEW] last_name                 → person.person_source_value (confidence=0.50)
  [AUTO] date_of_birth             → person.birth_datetime (confidence=0.95)
  [AUTO] gender                    → person.gender_concept_id (confidence=0.95)
  [AUTO] race                      → person.race_concept_id (confidence=0.95)
  [AUTO] ethnicity                 → person.ethnicity_concept_id (confidence=0.95)
  [AUTO] address                   → location.address_1 (confidence=0.95)
  [AUTO] city                      → location.city (confidence=0.95)
  [AUTO] state                     → location.state (confidence=0.95)
  [AUTO] zip_code                  → location.zip (confidence=0.95)


### 3b. Map Encounters to `Encounter` Resource

In [6]:
encounters_schema = project.map_schema(encounters)

print("Encounters → Encounter Resource")
print("=" * 70)
for item in encounters_schema.items:
    status = "[AUTO]" if item.confidence >= 0.85 else "[REVIEW]"
    print(f"  {status} {item.source_column:25s} → "
          f"{item.target_table}.{item.target_column} "
          f"(confidence={item.confidence:.2f})")

2026-04-16 00:28:21 [info     ] Stage 2: Schema mapping (local) embedding_model=cambridgeltl/SapBERT-from-PubMedBERT-fulltext embedding_provider=huggingface target_model=fhir_r4


2026-04-16 00:28:22 [info     ] local_schema_mapper.loading_model model=cambridgeltl/SapBERT-from-PubMedBERT-fulltext provider=huggingface target_standard=omop_cdm_v5.4


2026-04-16 00:28:22 [info     ] embedding_gateway.initialized  model=cambridgeltl/SapBERT-from-PubMedBERT-fulltext provider=huggingface


2026-04-16 00:28:22 [info     ] local_schema_mapper.computing_target_embeddings standard=omop_cdm_v5.4 targets=67


2026-04-16 00:28:22 [warning  ] local_schema_mapper.init_failed: sentence-transformers is required for HuggingFace embeddings. Install with: pip install sentence-transformers


2026-04-16 00:28:22 [info     ] Stage 2 complete               auto=7 review=0 total=8


2026-04-16 00:28:22 [info     ] local_storage.schema_mapping_saved items_count=8 project='FHIR End-to-End Demo'


2026-04-16 00:28:22 [info     ] project.schema_mapped          auto=7 project='FHIR End-to-End Demo' total=8


Encounters → Encounter Resource
  [AUTO] encounter_id              → visit_occurrence.visit_occurrence_id (confidence=0.95)
  [AUTO] patient_id                → person.person_id (confidence=0.95)
  [AUTO] encounter_type            → visit_occurrence.visit_concept_id (confidence=0.95)
  [AUTO] admission_date            → visit_occurrence.visit_start_date (confidence=0.95)
  [AUTO] discharge_date            → visit_occurrence.visit_end_date (confidence=0.95)
  [AUTO] facility                  → visit_occurrence.care_site_id (confidence=0.95)
  [REVIEW] department                → visit_occurrence.care_site_id (confidence=0.50)
  [AUTO] attending_provider        → visit_occurrence.provider_id (confidence=0.95)


### 3c. Map Diagnoses to `Condition` Resource

In [7]:
diagnoses_schema = project.map_schema(diagnoses)

print("Diagnoses → Condition Resource")
print("=" * 70)
for item in diagnoses_schema.items:
    status = "[AUTO]" if item.confidence >= 0.85 else "[REVIEW]"
    print(f"  {status} {item.source_column:25s} → "
          f"{item.target_table}.{item.target_column} "
          f"(confidence={item.confidence:.2f})")

2026-04-16 00:28:22 [info     ] Stage 2: Schema mapping (local) embedding_model=cambridgeltl/SapBERT-from-PubMedBERT-fulltext embedding_provider=huggingface target_model=fhir_r4


2026-04-16 00:28:22 [info     ] local_schema_mapper.loading_model model=cambridgeltl/SapBERT-from-PubMedBERT-fulltext provider=huggingface target_standard=omop_cdm_v5.4


2026-04-16 00:28:22 [info     ] embedding_gateway.initialized  model=cambridgeltl/SapBERT-from-PubMedBERT-fulltext provider=huggingface


2026-04-16 00:28:22 [info     ] local_schema_mapper.computing_target_embeddings standard=omop_cdm_v5.4 targets=67


2026-04-16 00:28:22 [warning  ] local_schema_mapper.init_failed: sentence-transformers is required for HuggingFace embeddings. Install with: pip install sentence-transformers


2026-04-16 00:28:22 [info     ] Stage 2 complete               auto=5 review=0 total=6


2026-04-16 00:28:22 [info     ] local_storage.schema_mapping_saved items_count=6 project='FHIR End-to-End Demo'


2026-04-16 00:28:22 [info     ] project.schema_mapped          auto=5 project='FHIR End-to-End Demo' total=6


Diagnoses → Condition Resource
  [AUTO] encounter_id              → visit_occurrence.visit_occurrence_id (confidence=0.95)
  [AUTO] patient_id                → person.person_id (confidence=0.95)
  [AUTO] diagnosis_code            → condition_occurrence.condition_source_value (confidence=0.95)
  [REVIEW] diagnosis_description     → condition_occurrence.condition_source_value (confidence=0.50)
  [AUTO] diagnosis_type            → condition_occurrence.condition_type_concept_id (confidence=0.95)
  [AUTO] diagnosis_date            → condition_occurrence.condition_start_date (confidence=0.95)


### 3d. Map Medications to `MedicationRequest` Resource

In [8]:
medications_schema = project.map_schema(medications)

print("Medications → MedicationRequest Resource")
print("=" * 70)
for item in medications_schema.items:
    status = "[AUTO]" if item.confidence >= 0.85 else "[REVIEW]"
    print(f"  {status} {item.source_column:25s} → "
          f"{item.target_table}.{item.target_column} "
          f"(confidence={item.confidence:.2f})")

2026-04-16 00:28:22 [info     ] Stage 2: Schema mapping (local) embedding_model=cambridgeltl/SapBERT-from-PubMedBERT-fulltext embedding_provider=huggingface target_model=fhir_r4


2026-04-16 00:28:22 [info     ] local_schema_mapper.loading_model model=cambridgeltl/SapBERT-from-PubMedBERT-fulltext provider=huggingface target_standard=omop_cdm_v5.4


2026-04-16 00:28:22 [info     ] embedding_gateway.initialized  model=cambridgeltl/SapBERT-from-PubMedBERT-fulltext provider=huggingface


2026-04-16 00:28:22 [info     ] local_schema_mapper.computing_target_embeddings standard=omop_cdm_v5.4 targets=67


2026-04-16 00:28:22 [warning  ] local_schema_mapper.init_failed: sentence-transformers is required for HuggingFace embeddings. Install with: pip install sentence-transformers


2026-04-16 00:28:22 [info     ] Stage 2 complete               auto=6 review=0 total=7


2026-04-16 00:28:22 [info     ] local_storage.schema_mapping_saved items_count=7 project='FHIR End-to-End Demo'


2026-04-16 00:28:22 [info     ] project.schema_mapped          auto=6 project='FHIR End-to-End Demo' total=7


Medications → MedicationRequest Resource
  [AUTO] prescription_id           → drug_exposure.drug_exposure_id (confidence=0.95)
  [AUTO] patient_id                → person.person_id (confidence=0.95)
  [AUTO] drug_code                 → drug_exposure.drug_source_value (confidence=0.95)
  [REVIEW] drug_name                 → drug_exposure.drug_source_value (confidence=0.50)
  [AUTO] quantity                  → drug_exposure.quantity (confidence=0.95)
  [AUTO] prescription_date         → drug_exposure.drug_exposure_start_date (confidence=0.95)
  [AUTO] prescriber_id             → drug_exposure.provider_id (confidence=0.95)


### 3e. Map Lab Results to `Observation` Resource

In [9]:
lab_results_schema = project.map_schema(lab_results)

print("Lab Results → Observation Resource")
print("=" * 70)
for item in lab_results_schema.items:
    status = "[AUTO]" if item.confidence >= 0.85 else "[REVIEW]"
    print(f"  {status} {item.source_column:25s} → "
          f"{item.target_table}.{item.target_column} "
          f"(confidence={item.confidence:.2f})")

2026-04-16 00:28:22 [info     ] Stage 2: Schema mapping (local) embedding_model=cambridgeltl/SapBERT-from-PubMedBERT-fulltext embedding_provider=huggingface target_model=fhir_r4


2026-04-16 00:28:22 [info     ] local_schema_mapper.loading_model model=cambridgeltl/SapBERT-from-PubMedBERT-fulltext provider=huggingface target_standard=omop_cdm_v5.4


2026-04-16 00:28:22 [info     ] embedding_gateway.initialized  model=cambridgeltl/SapBERT-from-PubMedBERT-fulltext provider=huggingface


2026-04-16 00:28:22 [info     ] local_schema_mapper.computing_target_embeddings standard=omop_cdm_v5.4 targets=67


2026-04-16 00:28:22 [warning  ] local_schema_mapper.init_failed: sentence-transformers is required for HuggingFace embeddings. Install with: pip install sentence-transformers


2026-04-16 00:28:22 [info     ] Stage 2 complete               auto=8 review=0 total=9


2026-04-16 00:28:22 [info     ] local_storage.schema_mapping_saved items_count=9 project='FHIR End-to-End Demo'


2026-04-16 00:28:22 [info     ] project.schema_mapped          auto=8 project='FHIR End-to-End Demo' total=9


Lab Results → Observation Resource
  [AUTO] lab_id                    → measurement.measurement_id (confidence=0.95)
  [AUTO] patient_id                → person.person_id (confidence=0.95)
  [AUTO] test_code                 → measurement.measurement_source_value (confidence=0.95)
  [REVIEW] test_name                 → measurement.measurement_source_value (confidence=0.50)
  [AUTO] result_value              → measurement.value_as_number (confidence=0.95)
  [AUTO] result_unit               → measurement.unit_source_value (confidence=0.95)
  [AUTO] reference_range           → measurement.range_low (confidence=0.95)
  [AUTO] result_flag               → measurement.value_as_concept_id (confidence=0.95)
  [AUTO] collection_date           → measurement.measurement_date (confidence=0.95)


### 3f. Review Schema Mapping Results

In [10]:
all_schema_maps = {
    "patients": patients_schema,
    "encounters": encounters_schema,
    "diagnoses": diagnoses_schema,
    "medications": medications_schema,
    "lab_results": lab_results_schema,
}

print("Schema Mapping Summary (FHIR R4)")
print("=" * 55)
for name, schema_map in all_schema_maps.items():
    total = len(schema_map.items)
    auto = sum(1 for item in schema_map.items if item.confidence >= 0.85)
    review = total - auto
    print(f"  {name:15s}: {auto}/{total} auto-mapped, {review} need review")

print("\nItems needing review:")
for name, schema_map in all_schema_maps.items():
    for item in schema_map.items:
        if item.confidence < 0.85:
            print(f"  [{name}] {item.source_column} → "
                  f"{item.target_table}.{item.target_column} "
                  f"(confidence={item.confidence:.2f})")

Schema Mapping Summary (FHIR R4)
  patients       : 10/11 auto-mapped, 1 need review
  encounters     : 7/8 auto-mapped, 1 need review
  diagnoses      : 5/6 auto-mapped, 1 need review
  medications    : 6/7 auto-mapped, 1 need review
  lab_results    : 8/9 auto-mapped, 1 need review

Items needing review:
  [patients] last_name → person.person_source_value (confidence=0.50)
  [encounters] department → visit_occurrence.care_site_id (confidence=0.50)
  [diagnoses] diagnosis_description → condition_occurrence.condition_source_value (confidence=0.50)
  [medications] drug_name → drug_exposure.drug_source_value (confidence=0.50)
  [lab_results] test_name → measurement.measurement_source_value (confidence=0.50)


## Stage 4: Concept Mapping

For FHIR, concept mapping translates source codes into standard terminologies.
The output uses FHIR `CodeableConcept` structures with proper CodeSystem URIs:

| Source Domain | Target Vocabulary | FHIR CodeSystem URI |
|---|---|---|
| Diagnoses (ICD-10) | SNOMED CT | `http://snomed.info/sct` |
| Medications (local) | RxNorm | `http://www.nlm.nih.gov/research/umls/rxnorm` |
| Lab tests (local) | LOINC | `http://loinc.org` |
| Procedures (CPT) | SNOMED CT | `http://snomed.info/sct` |

### 4a. Map Diagnosis Codes (ICD-10 → SNOMED CT)

In [11]:
diagnosis_codes = [
    {"code": "E11.9", "description": "Type 2 diabetes mellitus, without complications", "count": 1523},
    {"code": "I10", "description": "Essential (primary) hypertension", "count": 2891},
    {"code": "J18.9", "description": "Pneumonia, unspecified organism", "count": 456},
    {"code": "N18.6", "description": "End stage renal disease", "count": 234},
    {"code": "K21.0", "description": "Gastro-esophageal reflux disease with esophagitis", "count": 678},
    {"code": "M54.5", "description": "Low back pain", "count": 1345},
    {"code": "F32.9", "description": "Major depressive disorder, single episode", "count": 891},
    {"code": "J44.1", "description": "COPD with acute exacerbation", "count": 312},
]

dx_concept_map = project.map_concepts(
    codes=diagnosis_codes,
    vocabularies=["SNOMED", "ICD10CM"],
)

dx_summary = dx_concept_map.summary()
print("Diagnosis Concept Mapping (ICD-10 → SNOMED CT)")
print("=" * 50)
print(f"  Total:          {dx_summary['total']}")
print(f"  Auto-mapped:    {dx_summary['auto_mapped']}")
print(f"  Needs review:   {dx_summary['needs_review']}")
print(f"  Auto rate:      {dx_summary['auto_rate']:.1%}")
print(f"  Coverage:       {dx_summary['coverage']:.1%}")

2026-04-16 00:28:22 [info     ] Stage 3: Concept mapping (local) columns=[] knowledge_backend=bm25s


2026-04-16 00:28:22 [info     ] knowledge.creating_backend     backend=bm25s path=example_assets/vocabulary_download_v5/indexes/concepts.json


Split strings:   0%|          | 0/623910 [00:00<?, ?it/s]

Stem Tokens:   0%|          | 0/623910 [00:00<?, ?it/s]

BM25S Count Tokens:   0%|          | 0/623910 [00:00<?, ?it/s]

...

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

2026-04-16 00:28:31 [info     ] Stage 3 complete               auto_rate=100.0% total_codes=8


2026-04-16 00:28:31 [info     ] local_storage.concept_mapping_saved items_count=8 project='FHIR End-to-End Demo'


2026-04-16 00:28:31 [info     ] project.concepts_mapped        auto_rate=100.0 project='FHIR End-to-End Demo' total=8


Diagnosis Concept Mapping (ICD-10 → SNOMED CT)
  Total:          8
  Auto-mapped:    8
  Needs review:   0
  Auto rate:      10000.0%
  Coverage:       10000.0%


### 4b. Map Medication Codes (Local → RxNorm)

In [12]:
medication_codes = [
    {"code": "MET500", "description": "Metformin 500mg oral tablet", "count": 2300},
    {"code": "LISIN10", "description": "Lisinopril 10mg oral tablet", "count": 1850},
    {"code": "ATOR20", "description": "Atorvastatin 20mg oral tablet", "count": 1600},
    {"code": "AMOX500", "description": "Amoxicillin 500mg oral capsule", "count": 900},
    {"code": "OMEP20", "description": "Omeprazole 20mg oral capsule", "count": 1100},
    {"code": "ASA81", "description": "Aspirin 81mg oral tablet", "count": 3200},
    {"code": "HYDRO25", "description": "Hydrochlorothiazide 25mg oral tablet", "count": 950},
]

rx_concept_map = project.map_concepts(
    codes=medication_codes,
    vocabularies=["RxNorm"],
)

rx_summary = rx_concept_map.summary()
print("Medication Concept Mapping (Local → RxNorm)")
print("=" * 50)
print(f"  Total:          {rx_summary['total']}")
print(f"  Auto-mapped:    {rx_summary['auto_mapped']}")
print(f"  Needs review:   {rx_summary['needs_review']}")
print(f"  Auto rate:      {rx_summary['auto_rate']:.1%}")
print(f"  Coverage:       {rx_summary['coverage']:.1%}")

2026-04-16 00:28:31 [info     ] Stage 3: Concept mapping (local) columns=[] knowledge_backend=bm25s


2026-04-16 00:28:31 [info     ] knowledge.creating_backend     backend=bm25s path=example_assets/vocabulary_download_v5/indexes/concepts.json


Split strings:   0%|          | 0/623910 [00:00<?, ?it/s]

Stem Tokens:   0%|          | 0/623910 [00:00<?, ?it/s]

BM25S Count Tokens:   0%|          | 0/623910 [00:00<?, ?it/s]

...

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

2026-04-16 00:28:39 [info     ] Stage 3 complete               auto_rate=100.0% total_codes=7


2026-04-16 00:28:39 [info     ] local_storage.concept_mapping_saved items_count=7 project='FHIR End-to-End Demo'


2026-04-16 00:28:39 [info     ] project.concepts_mapped        auto_rate=100.0 project='FHIR End-to-End Demo' total=7


Medication Concept Mapping (Local → RxNorm)
  Total:          7
  Auto-mapped:    7
  Needs review:   0
  Auto rate:      10000.0%
  Coverage:       10000.0%


### 4c. Map Lab Test Codes (Local → LOINC)

In [13]:
lab_codes = [
    {"code": "GLU", "description": "Glucose, serum", "count": 4500},
    {"code": "HBA1C", "description": "Hemoglobin A1c", "count": 2100},
    {"code": "BUN", "description": "Blood urea nitrogen", "count": 3200},
    {"code": "CREAT", "description": "Creatinine, serum", "count": 3100},
    {"code": "WBC", "description": "White blood cell count", "count": 5600},
    {"code": "HGB", "description": "Hemoglobin", "count": 5200},
    {"code": "PLT", "description": "Platelet count", "count": 4800},
    {"code": "TSH", "description": "Thyroid stimulating hormone", "count": 1900},
    {"code": "ALT", "description": "Alanine aminotransferase", "count": 2800},
    {"code": "SODIUM", "description": "Sodium, serum", "count": 3800},
]

lab_concept_map = project.map_concepts(
    codes=lab_codes,
    vocabularies=["LOINC"],
)

lab_summary = lab_concept_map.summary()
print("Lab Test Concept Mapping (Local → LOINC)")
print("=" * 50)
print(f"  Total:          {lab_summary['total']}")
print(f"  Auto-mapped:    {lab_summary['auto_mapped']}")
print(f"  Needs review:   {lab_summary['needs_review']}")
print(f"  Auto rate:      {lab_summary['auto_rate']:.1%}")
print(f"  Coverage:       {lab_summary['coverage']:.1%}")

2026-04-16 00:28:40 [info     ] Stage 3: Concept mapping (local) columns=[] knowledge_backend=bm25s


2026-04-16 00:28:40 [info     ] knowledge.creating_backend     backend=bm25s path=example_assets/vocabulary_download_v5/indexes/concepts.json


Split strings:   0%|          | 0/623910 [00:00<?, ?it/s]

Stem Tokens:   0%|          | 0/623910 [00:00<?, ?it/s]

BM25S Count Tokens:   0%|          | 0/623910 [00:00<?, ?it/s]

...

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

2026-04-16 00:28:48 [info     ] Stage 3 complete               auto_rate=100.0% total_codes=10


2026-04-16 00:28:48 [info     ] local_storage.concept_mapping_saved items_count=10 project='FHIR End-to-End Demo'


2026-04-16 00:28:48 [info     ] project.concepts_mapped        auto_rate=100.0 project='FHIR End-to-End Demo' total=10


Lab Test Concept Mapping (Local → LOINC)
  Total:          10
  Auto-mapped:    10
  Needs review:   0
  Auto rate:      10000.0%
  Coverage:       10000.0%


### 4d. Review and Finalize Concept Mappings

Review items flagged for human attention across all three domains.

In [14]:
concept_maps = {
    "Diagnoses (ICD-10 → SNOMED)": dx_concept_map,
    "Medications (Local → RxNorm)": rx_concept_map,
    "Lab Tests (Local → LOINC)": lab_concept_map,
}

for domain, cmap in concept_maps.items():
    review_items = list(cmap.needs_review())
    if review_items:
        print(f"{domain} — {len(review_items)} items need review:")
        print("-" * 70)
        for item in review_items:
            print(f"  {item.source_code}: {item.source_description}")
            print(f"    → {item.target_concept_name} (confidence={item.confidence:.2f})")
            if item.candidates:
                for i, c in enumerate(item.candidates[:3]):
                    marker = " <<" if i == 0 else ""
                    print(f"    [{i}] {c.concept_name} ({c.vocabulary_id}, "
                          f"score={c.score:.3f}){marker}")
            print()
    else:
        print(f"{domain} — all items auto-mapped.")
    print()

Diagnoses (ICD-10 → SNOMED) — all items auto-mapped.

Medications (Local → RxNorm) — all items auto-mapped.

Lab Tests (Local → LOINC) — all items auto-mapped.



In [15]:
# Approve all review items and finalize
for domain, cmap in concept_maps.items():
    cmap.approve_all()
    cmap.finalize()

print("All concept mappings finalized.")
for domain, cmap in concept_maps.items():
    s = cmap.summary()
    print(f"  {domain}: {s['auto_mapped']}/{s['total']} mapped, "
          f"coverage={s['coverage']:.1%}")

All concept mappings finalized.
  Diagnoses (ICD-10 → SNOMED): 8/8 mapped, coverage=10000.0%
  Medications (Local → RxNorm): 7/7 mapped, coverage=10000.0%
  Lab Tests (Local → LOINC): 10/10 mapped, coverage=10000.0%


## Stage 5: Run ETL Transformation

For FHIR targets, the ETL process generates **FHIR R4 Bundle JSON** files
instead of CSV tables. Each source file produces a Bundle containing the
corresponding FHIR resources with:

- Proper `resourceType` declarations
- `CodeableConcept` fields with correct CodeSystem URIs
- Resource references (`Patient/123`, `Encounter/456`, etc.)
- FHIR-required fields populated (e.g., `status`, `intent`)

In [16]:
# ETL: patients → Patient resources
patients_etl = project.run_etl(
    patients,
    output_dir="./fhir_output",
    schema_mapping=patients_schema,
    concept_mapping=dx_concept_map,
)
print("Patients ETL completed → Patient Bundle")

2026-04-16 00:28:48 [info     ] Reading source                 format=csv path=example_assets/09_end_to_end_fhir/patients.csv


2026-04-16 00:28:48 [info     ] Processing table               columns=6 table=person


2026-04-16 00:28:48 [info     ] Processing table               columns=4 table=location


2026-04-16 00:28:48 [info     ] ETL complete                   duration=0.00458 success=True


2026-04-16 00:28:48 [info     ] project.etl_complete           output_dir=./fhir_output project='FHIR End-to-End Demo'


Patients ETL completed → Patient Bundle


In [17]:
# ETL: encounters → Encounter resources
encounters_etl = project.run_etl(
    encounters,
    output_dir="./fhir_output",
    schema_mapping=encounters_schema,
    concept_mapping=dx_concept_map,
)
print("Encounters ETL completed → Encounter Bundle")

2026-04-16 00:28:48 [info     ] Reading source                 format=csv path=example_assets/09_end_to_end_fhir/encounters.csv


2026-04-16 00:28:48 [info     ] Processing table               columns=6 table=visit_occurrence


2026-04-16 00:28:48 [info     ] Processing table               columns=1 table=person


2026-04-16 00:28:48 [info     ] ETL complete                   duration=0.002581 success=True


2026-04-16 00:28:48 [info     ] project.etl_complete           output_dir=./fhir_output project='FHIR End-to-End Demo'


Encounters ETL completed → Encounter Bundle


In [18]:
# ETL: diagnoses → Condition resources
diagnoses_etl = project.run_etl(
    diagnoses,
    output_dir="./fhir_output",
    schema_mapping=diagnoses_schema,
    concept_mapping=dx_concept_map,
)
print("Diagnoses ETL completed → Condition Bundle")

2026-04-16 00:28:48 [info     ] Reading source                 format=csv path=example_assets/09_end_to_end_fhir/diagnoses.csv


2026-04-16 00:28:48 [info     ] Processing table               columns=1 table=visit_occurrence


2026-04-16 00:28:48 [info     ] Processing table               columns=1 table=person


2026-04-16 00:28:48 [info     ] Processing table               columns=3 table=condition_occurrence


2026-04-16 00:28:48 [info     ] ETL complete                   duration=0.003418 success=True


2026-04-16 00:28:48 [info     ] project.etl_complete           output_dir=./fhir_output project='FHIR End-to-End Demo'


Diagnoses ETL completed → Condition Bundle


In [19]:
# ETL: medications → MedicationRequest resources
medications_etl = project.run_etl(
    medications,
    output_dir="./fhir_output",
    schema_mapping=medications_schema,
    concept_mapping=rx_concept_map,
)
print("Medications ETL completed → MedicationRequest Bundle")

2026-04-16 00:28:48 [info     ] Reading source                 format=csv path=example_assets/09_end_to_end_fhir/medications.csv


2026-04-16 00:28:48 [info     ] Processing table               columns=5 table=drug_exposure


2026-04-16 00:28:48 [info     ] Processing table               columns=1 table=person


2026-04-16 00:28:48 [info     ] ETL complete                   duration=0.002692 success=True


2026-04-16 00:28:48 [info     ] project.etl_complete           output_dir=./fhir_output project='FHIR End-to-End Demo'


Medications ETL completed → MedicationRequest Bundle


In [20]:
# ETL: lab results → Observation resources
lab_results_etl = project.run_etl(
    lab_results,
    output_dir="./fhir_output",
    schema_mapping=lab_results_schema,
    concept_mapping=lab_concept_map,
)
print("Lab Results ETL completed → Observation Bundle")

2026-04-16 00:28:48 [info     ] Reading source                 format=csv path=example_assets/09_end_to_end_fhir/lab_results.csv


2026-04-16 00:28:48 [info     ] Processing table               columns=7 table=measurement


2026-04-16 00:28:48 [info     ] Processing table               columns=1 table=person


2026-04-16 00:28:48 [info     ] ETL complete                   duration=0.002738 success=True


2026-04-16 00:28:48 [info     ] project.etl_complete           output_dir=./fhir_output project='FHIR End-to-End Demo'


Lab Results ETL completed → Observation Bundle


## Stage 6: Validate Output

FHIR validation checks the generated Bundle JSON against R4 profiles:

- Required fields present for each resource type
- Valid `resourceType` declarations
- CodeableConcept fields have valid CodeSystem URIs
- Resource references resolve within the Bundle
- Status fields use valid FHIR value sets
- Date/time formats conform to FHIR specifications

In [21]:
# Validate all ETL results
etl_results = [
    ("patients → Patient", patients_etl),
    ("encounters → Encounter", encounters_etl),
    ("diagnoses → Condition", diagnoses_etl),
    ("medications → MedicationRequest", medications_etl),
    ("lab_results → Observation", lab_results_etl),
]

print("FHIR R4 Validation Results")
print("=" * 60)
all_passed = True
for name, result in etl_results:
    v = project.validate(etl_result=result)
    status = "PASS" if v['all_passed'] else "FAIL"
    all_passed = all_passed and v['all_passed']
    print(f"  {name:40s} {status}")
    if not v['all_passed']:
        for table in v['tables']:
            if not table.get('passed'):
                for error in table.get('errors', []):
                    print(f"    - {error}")

print(f"\nOverall: {'ALL PASSED' if all_passed else 'SOME FAILED'}")

FHIR R4 Validation Results


Calculating Metrics: 0it [00:00, ?it/s]

2026-04-16 00:28:48 [info     ] gx_validator.validated         completeness=1.00 conformance=1.00 passed=True plausibility=1.00 table=drug_exposure


Calculating Metrics: 0it [00:00, ?it/s]

2026-04-16 00:28:49 [info     ] gx_validator.validated         completeness=1.00 conformance=1.00 passed=True plausibility=1.00 table=location


...

2026-04-16 00:28:50 [info     ] gx_validator.validated         completeness=1.00 conformance=1.00 passed=True plausibility=1.00 table=visit_occurrence


Calculating Metrics: 0it [00:00, ?it/s]

2026-04-16 00:28:50 [info     ] gx_validator.validated         completeness=1.00 conformance=1.00 passed=True plausibility=1.00 table=person


2026-04-16 00:28:50 [info     ] project.validated              all_passed=True project='FHIR End-to-End Demo' tables=6


  lab_results → Observation                PASS

Overall: ALL PASSED


## Stage 7: Inspect FHIR Output

Let us examine the generated FHIR Bundles to verify the transformation
and inspect the resource structure.

In [22]:
import json
import os

output_dir = "./fhir_output"

if os.path.exists(output_dir):
    files = sorted(os.listdir(output_dir))
    print(f"Generated FHIR files in {output_dir}:")
    print("-" * 50)
    for f in files:
        filepath = os.path.join(output_dir, f)
        size_kb = os.path.getsize(filepath) / 1024
        print(f"  {f:40s} {size_kb:8.1f} KB")
else:
    print(f"Output directory {output_dir} does not exist yet.")
    print("Run the ETL cells above first.")

Generated FHIR files in ./fhir_output:
--------------------------------------------------
  condition_occurrence.csv                      0.6 KB
  drug_exposure.csv                             0.6 KB
  location.csv                                  0.5 KB
  measurement.csv                               0.9 KB
  person.csv                                    0.1 KB
  visit_occurrence.csv                          0.1 KB


In [23]:
# Inspect a Patient resource from the Bundle
patient_bundle_path = os.path.join(output_dir, "Patient.json")
if os.path.exists(patient_bundle_path):
    with open(patient_bundle_path) as f:
        patient_bundle = json.load(f)

    print(f"Patient Bundle: {patient_bundle.get('resourceType', 'unknown')}")
    print(f"Total entries: {len(patient_bundle.get('entry', []))}")
    print()

    # Show the first Patient resource
    if patient_bundle.get('entry'):
        first_patient = patient_bundle['entry'][0].get('resource', {})
        print("First Patient resource:")
        print(json.dumps(first_patient, indent=2))
else:
    print("Patient.json not found. Run ETL cells above first.")

Patient.json not found. Run ETL cells above first.


In [24]:
# Inspect a Condition resource (diagnosis with SNOMED coding)
condition_bundle_path = os.path.join(output_dir, "Condition.json")
if os.path.exists(condition_bundle_path):
    with open(condition_bundle_path) as f:
        condition_bundle = json.load(f)

    print(f"Condition Bundle: {condition_bundle.get('resourceType', 'unknown')}")
    print(f"Total entries: {len(condition_bundle.get('entry', []))}")
    print()

    # Show the first Condition resource with CodeableConcept
    if condition_bundle.get('entry'):
        first_condition = condition_bundle['entry'][0].get('resource', {})
        print("First Condition resource:")
        print(json.dumps(first_condition, indent=2))
        print()

        # Highlight the CodeableConcept structure
        code = first_condition.get('code', {})
        if code.get('coding'):
            coding = code['coding'][0]
            print("CodeableConcept breakdown:")
            print(f"  system:  {coding.get('system')}")
            print(f"  code:    {coding.get('code')}")
            print(f"  display: {coding.get('display')}")
else:
    print("Condition.json not found. Run ETL cells above first.")

Condition.json not found. Run ETL cells above first.


In [25]:
# Inspect an Observation resource (lab result with LOINC coding + valueQuantity)
observation_bundle_path = os.path.join(output_dir, "Observation.json")
if os.path.exists(observation_bundle_path):
    with open(observation_bundle_path) as f:
        observation_bundle = json.load(f)

    print(f"Observation Bundle: {observation_bundle.get('resourceType', 'unknown')}")
    print(f"Total entries: {len(observation_bundle.get('entry', []))}")
    print()

    # Show the first Observation resource
    if observation_bundle.get('entry'):
        first_obs = observation_bundle['entry'][0].get('resource', {})
        print("First Observation resource:")
        print(json.dumps(first_obs, indent=2))
        print()

        # Highlight code and value
        code = first_obs.get('code', {})
        value = first_obs.get('valueQuantity', {})
        if code.get('coding'):
            coding = code['coding'][0]
            print(f"Lab test: {coding.get('display')} (LOINC {coding.get('code')})")
        if value:
            print(f"Value: {value.get('value')} {value.get('unit')}")
else:
    print("Observation.json not found. Run ETL cells above first.")

Observation.json not found. Run ETL cells above first.


In [26]:
# Inspect a MedicationRequest resource (with RxNorm coding)
medrx_bundle_path = os.path.join(output_dir, "MedicationRequest.json")
if os.path.exists(medrx_bundle_path):
    with open(medrx_bundle_path) as f:
        medrx_bundle = json.load(f)

    print(f"MedicationRequest Bundle: {medrx_bundle.get('resourceType', 'unknown')}")
    print(f"Total entries: {len(medrx_bundle.get('entry', []))}")
    print()

    if medrx_bundle.get('entry'):
        first_medrx = medrx_bundle['entry'][0].get('resource', {})
        print("First MedicationRequest resource:")
        print(json.dumps(first_medrx, indent=2))
        print()

        med = first_medrx.get('medicationCodeableConcept', {})
        if med.get('coding'):
            coding = med['coding'][0]
            print(f"Medication: {coding.get('display')} (RxNorm {coding.get('code')})")
else:
    print("MedicationRequest.json not found. Run ETL cells above first.")

MedicationRequest.json not found. Run ETL cells above first.


## Export: Source-to-Concept Map

Even for FHIR targets, the `source_to_concept_map` export is useful for
documenting the code translation decisions. This is particularly valuable
for audit trails and reproducibility.

In [27]:
import pandas as pd

# Combine all concept maps
dx_stcm = dx_concept_map.to_source_to_concept_map()
rx_stcm = rx_concept_map.to_source_to_concept_map()
lab_stcm = lab_concept_map.to_source_to_concept_map()

all_stcm = dx_stcm + rx_stcm + lab_stcm
stcm_df = pd.DataFrame(all_stcm)

print(f"Combined source_to_concept_map: {len(stcm_df)} entries")
print(f"  Diagnoses:   {len(dx_stcm)} mappings")
print(f"  Medications: {len(rx_stcm)} mappings")
print(f"  Lab tests:   {len(lab_stcm)} mappings")
print()
print(stcm_df.to_string(index=False))

# Save
stcm_path = os.path.join(output_dir, "source_to_concept_map.csv")
stcm_df.to_csv(stcm_path, index=False)
print(f"\nSaved to {stcm_path}")

Combined source_to_concept_map: 25 entries
  Diagnoses:   8 mappings
  Medications: 7 mappings
  Lab tests:   10 mappings

source_code  source_concept_id source_vocabulary_id                                source_description                           source_code_description  target_concept_id                                                                               target_concept_name target_vocabulary_id  confidence method valid_start_date valid_end_date invalid_reason
      E11.9                  0             Hospital   Type 2 diabetes mellitus, without complications   Type 2 diabetes mellitus, without complications           44787902                                               Lipoatrophic diabetes mellitus without complication               SNOMED   10.359390   auto       1970-01-01     2099-12-31           None
        I10                  0             Hospital                  Essential (primary) hypertension                  Essential (primary) hypertension             3

## Summary

This notebook demonstrated the complete Portiere pipeline for transforming hospital
data into **FHIR R4 resources**. The key stages were:

1. **Configuration** — Set `target_model="fhir_r4"` to target FHIR instead of OMOP
2. **Ingestion** — Loaded five source files (patients, encounters, diagnoses, medications, lab results)
3. **Schema Mapping** — Mapped source columns to FHIR resource fields (e.g., `Patient.birthDate`, `Condition.code`)
4. **Concept Mapping** — Translated codes to SNOMED CT, RxNorm, and LOINC with proper CodeSystem URIs
5. **ETL Execution** — Generated FHIR Bundle JSON files with valid resource structures
6. **Validation** — Verified resources against FHIR R4 profiles

### FHIR R4 Resource Reference

The resources generated in this pipeline:

| FHIR Resource | Description | Required Fields |
|---|---|---|
| `Patient` | Patient demographics | `id`, `gender` |
| `Encounter` | Hospital visits / admissions | `id`, `status`, `class`, `subject` |
| `Condition` | Diagnoses and problems | `id`, `code`, `subject` |
| `MedicationRequest` | Medication orders / prescriptions | `id`, `status`, `intent`, `medicationCodeableConcept`, `subject` |
| `Observation` | Lab results, vitals, assessments | `id`, `status`, `code`, `subject` |
| `Procedure` | Surgical / diagnostic procedures | `id`, `status`, `code`, `subject` |
| `AllergyIntolerance` | Allergies and adverse reactions | `id`, `patient` |
| `DiagnosticReport` | Lab / radiology reports | `id`, `status`, `code`, `subject` |

### FHIR Terminology System URIs

| Vocabulary | FHIR CodeSystem URI |
|---|---|
| SNOMED CT | `http://snomed.info/sct` |
| LOINC | `http://loinc.org` |
| RxNorm | `http://www.nlm.nih.gov/research/umls/rxnorm` |
| ICD-10-CM | `http://hl7.org/fhir/sid/icd-10-cm` |
| CPT-4 | `http://www.ama-assn.org/go/cpt` |
| NDC | `http://hl7.org/fhir/sid/ndc` |
| UCUM (units) | `http://unitsofmeasure.org` |

### Next Steps

1. **Load into a FHIR Server** — Upload Bundles to HAPI FHIR, Azure FHIR, Google Cloud Healthcare,
   or AWS HealthLake.

2. **FHIR Validation** — Run the [HL7 FHIR Validator](https://confluence.hl7.org/display/FHIR/Using+the+FHIR+Validator)
   for profile-level validation beyond basic structural checks.

3. **SMART on FHIR Apps** — Connect patient-facing or clinician-facing SMART apps
   to the FHIR server.

4. **Bulk Data Export** — Use FHIR Bulk Data Access (`$export`) for analytics pipelines.

5. **US Core Profiles** — For US healthcare, validate against
   [US Core Implementation Guide](http://hl7.org/fhir/us/core/) profiles.

6. **Compare with OMOP** — Run Notebook 08 (`08_end_to_end_omop.ipynb`) on the same
   source data to compare OMOP CDM output side by side.